# Trace Dump


In [ ]:
from mlflow import MlflowClient

experiment_id = "766133932460693192"

client = MlflowClient()
traces = client.search_traces(experiment_ids=[experiment_id], filter_string="request_id = 'fe107e3f03004252ae919f790c6c67ae'")

** in UI, it need to interface with mlflow for available trace to pull

In [ ]:
# Convert traces to JSON and save to file
import json
from datetime import datetime

# Create a directory for logs if it doesn't exist
os.makedirs('logs', exist_ok=True)

# Generate a timestamp for the filename
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'logs/traces_{timestamp}.json'

# Convert traces to JSON and save to file
with open(filename, 'w') as f:
    json.dump(traces[0].to_json(), f, indent=2, default=str)

print(f"Traces saved to {filename}")


# Trace To Detailed Graph


In [1]:
import os
from trace_to_graph_crewai import TraceGraphFromCrewAI
import json 

In [2]:
def run_trace_to_graph(trace_file: str):
 
    # Create TraceGraph instance and process the trace
    trace_graph = TraceGraphFromCrewAI(trace_file)
    
    # Generate the detailed graph with relationships
    detailed_graph = trace_graph.generate_detailed_knowledge_graph()
    
    current_dir = os.getcwd()

    # Save the results
    output_dir = os.path.join(current_dir, 'data', 'output')
    os.makedirs(output_dir, exist_ok=True)
    
    # Save detailed graph
    with open(os.path.join(output_dir, 'detailed_graph_new_format.json'), 'w') as f:
        json.dump(detailed_graph, f, indent=2)

In [3]:
run_trace_to_graph("./data/input/mlflow_trace.json")

# Jailbreak Each Process


In [3]:
import os
import langchain_openai
from dotenv import load_dotenv
from static_jailbreak_injection_crewai import JailbreakInjection

In [4]:
load_dotenv()

def run_jailbreak():
    # Initialize the LLM for jailbreak testing
    llm = langchain_openai.ChatOpenAI(
        model="deepseek/deepseek-chat",
        openai_api_key=os.getenv("DEEPSEEK_OPENROUTER_API_KEY"),
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0.2
    )

    # Initialize the jailbreak test with the graph data
    jailbreaking_test = JailbreakInjection(llm, './data/output/detailed_graph_new_format.json')

    # Run the jailbreak tests
    jailbreaking_test.run_jailbreaking_injection_static_test(jailbreak_test_attempts=1)

run_jailbreak()

loaded 37 jailbreak prompts


Testing processes for static jailbreak injection:  31%|███▏      | 5/16 [02:35<05:41, 31.03s/it]


KeyboardInterrupt: 

# Add jailbreak ASR to process and risk level to component 